# Capítulo 1: Línea de Comandos — Laboratorio práctico
**Tópicos Avanzados para Data Science en Python**

En este notebook practicaremos los comandos de la **Sesión 1.1 (Comandos y manipulación de archivos)** y la **Sesión 1.2 (Scripts de Shell y Automatización)** directamente en Google Colab.

| Tema | Comandos |
|---|---|
| Entorno | `uname`, `pwd`, `ls` |
| Búsqueda | `grep` |
| Manipulación | `cp`, `mv`, `rm` |
| Localización | `find` |
| Automatización | scripts `.sh` (shebang, comentarios, comandos) |

> **Recuerda:** En Colab trabajamos sobre **Linux real** (Ubuntu), no una emulación.

## 0. ¿Cómo ejecutar comandos de shell en Colab?

| Forma | Ejemplo | Cuándo usarla |
|---|---|---|
| `!comando` | `!ls -lh` | Un comando suelto (se ejecuta en una sub-shell) |
| `%%bash` | (primera línea de la celda) | Varias líneas de bash en una sola celda |
| `%cd ruta` | `%cd /content` | Cambiar de directorio **de forma permanente** |

⚠️ `!cd carpeta` **no** cambia el directorio para las siguientes celdas, porque cada `!` abre una shell nueva. Usa `%cd`.

In [ ]:
# ¿Qué sistema operativo tiene Colab?
!uname -a

In [ ]:
# Distribución de Linux y shell disponible
!cat /etc/os-release | head -4
!echo "Shell: $SHELL"
!which bash python3 grep find

## 1. Preparando el laboratorio (dataset de práctica)
Ejecuta la siguiente celda para crear un mini *data lake* con CSVs, logs de un pipeline y archivos temporales.

In [ ]:
%%bash
# Partimos de cero
rm -rf /content/lab_shell
mkdir -p /content/lab_shell/{raw,logs,tmp,processed}
cd /content/lab_shell

# --- Datasets de ventas ---
cat > raw/ventas_2024.csv <<'EOF'
id,fecha,cliente,producto,monto,estado
1,2024-01-15,ana@mail.com,Laptop,3500,OK
2,2024-02-03,luis@mail.com,Mouse,45,OK
3,2024-02-20,maria@mail.com,Monitor,800,ERROR
4,2024-03-11,pedro@mail.com,Teclado,120,OK
5,2024-04-01,sofia@mail.com,Laptop,3600,PENDIENTE
6,2024-05-09,ana@mail.com,Audifonos,250,OK
7,2024-06-18,jorge@mail.com,Monitor,820,ERROR
8,2024-07-22,carla@mail.com,Mouse,50,OK
EOF

cat > raw/ventas_2025.csv <<'EOF'
id,fecha,cliente,producto,monto,estado
1,2025-01-05,luis@mail.com,Laptop,3700,OK
2,2025-01-19,maria@mail.com,Tablet,1500,OK
3,2025-02-14,pedro@mail.com,Mouse,40,error
4,2025-03-02,correo_invalido,Monitor,850,OK
5,2025-03-28,sofia@mail.com,Teclado,130,OK
6,2025-04-10,jorge@mail.com,Laptop,3650,PENDIENTE
EOF

cat > raw/Clientes.CSV <<'EOF'
cliente,ciudad,segmento
ana@mail.com,Lima,Premium
luis@mail.com,Arequipa,Standard
maria@mail.com,Cusco,Premium
pedro@mail.com,Lima,Standard
EOF

# --- Logs del pipeline ETL ---
cat > logs/etl_lunes.log <<'EOF'
2025-03-03 08:00:01 INFO  Inicio del pipeline
2025-03-03 08:00:05 INFO  Leyendo raw/ventas_2025.csv
2025-03-03 08:00:09 WARN  Columna monto con valores nulos
2025-03-03 08:00:12 ERROR Fallo conexion a base de datos
2025-03-03 08:00:20 INFO  Reintentando conexion
2025-03-03 08:00:25 INFO  Pipeline finalizado
EOF

cat > logs/etl_martes.log <<'EOF'
2025-03-04 08:00:01 INFO  Inicio del pipeline
2025-03-04 08:00:07 error Timeout leyendo archivo
2025-03-04 08:00:10 ERROR Esquema invalido en ventas_2025.csv
2025-03-04 08:00:30 INFO  Pipeline finalizado
EOF

mkdir -p logs/archivo_2024
cat > logs/archivo_2024/etl_diciembre.log <<'EOF'
2024-12-30 08:00:01 INFO  Inicio del pipeline
2024-12-30 08:00:15 ERROR Disco lleno
2024-12-30 08:00:16 INFO  Pipeline finalizado
EOF

# --- Archivos temporales y ocultos ---
echo "cache"      > tmp/cache_1.tmp
echo "cache"      > tmp/cache_2.tmp
echo "parcial"    > tmp/staging_parcial.csv
echo "API_KEY=123" > .env

# Un archivo "grande" (~1 MB) para practicar find -size
seq 1 150000 > raw/big_dataset.txt

# Un log "viejo" (fecha modificada a hace 60 días) para find -mtime
touch -d "60 days ago" logs/archivo_2024/etl_diciembre.log

echo "Laboratorio listo en: $(pwd)"

## 2. `pwd` y `ls` — ¿Dónde estoy y qué hay?
- `pwd`: muestra la ruta absoluta del directorio actual.
- `ls`: lista archivos y carpetas.

| Opción | Significado |
|---|---|
| `-l` | Detalle (permisos, tamaño, fecha) |
| `-a` | Incluye archivos ocultos (los que empiezan con `.`) |
| `-h` | Tamaños legibles (K, M, G) |

In [ ]:
%cd /content/lab_shell
!pwd

In [ ]:
!ls

In [ ]:
!ls -l raw

In [ ]:
# Compara: ¿qué archivo nuevo aparece con -a?
!ls -lah

### ✏️ Ejercicio 2.1
1. Lista **en detalle y con tamaños legibles** el contenido de la carpeta `raw`. ¿Cuál es el archivo más pesado?
2. Muestra **todos** los archivos (incluidos ocultos) de la carpeta actual. ¿Qué archivo oculto hay y por qué sería peligroso subirlo a un repositorio?
3. Lista solo los archivos `.log` que están en la carpeta `logs`.

In [ ]:
# Tu código aquí

## 3. `grep` — Buscar patrones en archivos de texto
Permite **filtrar registros sin cargar todo a memoria**. Muy útil para logs, detección de fallos y control de calidad.

| Comando | ¿Qué hace? | Uso DS/DE |
|---|---|---|
| `grep palabra file` | Busca texto | Logs |
| `grep -i` | Ignora mayúsculas | Normalización |
| `grep -n` | Nº de línea | Debug |
| `grep -c` | Conteo | Métricas |
| `grep -v` | Excluir | Limpieza |
| `grep -r` | Recursivo | Data lakes |
| `grep -E` | Expresiones regulares | Validación |

In [ ]:
# Buscar texto
!grep ERROR logs/etl_lunes.log

In [ ]:
# -i: ignora mayúsculas/minúsculas (encuentra 'ERROR' y 'error')
!grep -i error logs/etl_martes.log

In [ ]:
# -n: muestra el número de línea (útil para debug)
!grep -n Laptop raw/ventas_2024.csv

In [ ]:
# -c: cuenta coincidencias (métrica rápida)
!grep -c OK raw/ventas_2024.csv

In [ ]:
# -v: excluye líneas (ej. quitar el encabezado del CSV)
!grep -v "^id," raw/ventas_2024.csv

In [ ]:
# -r: búsqueda recursiva en toda una carpeta (data lake)
!grep -r ERROR logs/

In [ ]:
# -E: expresiones regulares extendidas -> ventas de Laptop O Monitor
!grep -E "Laptop|Monitor" raw/ventas_2024.csv

### ✏️ Ejercicio 3.1 — Análisis de logs
1. ¿Cuántas líneas con `ERROR` (en mayúsculas o minúsculas) hay en `logs/etl_martes.log`?
2. Muestra, con su número de línea, los `WARN` de `logs/etl_lunes.log`.
3. Busca **recursivamente** todos los errores (sin importar mayúsculas) en la carpeta `logs` mostrando el número de línea.
4. Guarda en `processed/errores.txt` todas las líneas de error de todos los logs. *(Pista: redirección `>`)*

In [ ]:
# Tu código aquí

### ✏️ Ejercicio 3.2 — Control de calidad de datos
1. Muestra las ventas de `raw/ventas_2025.csv` **sin el encabezado**.
2. Encuentra los registros cuyo `estado` **no** sea `OK` (excluye también el encabezado).
3. Usa `grep -E` para detectar en `raw/ventas_2025.csv` los registros con un correo **inválido** (que no tengan el formato `algo@algo.com`).
4. ¿Cuántas ventas de 2024 se hicieron en el **primer trimestre** (meses 01, 02 o 03)?

In [ ]:
# Tu código aquí

## 4. `cp` — Copiar archivos y directorios
Respaldar datasets, crear versiones **antes de transformar** datos, preparar entornos de prueba.

| Comando | ¿Qué hace? | Uso DS/DE |
|---|---|---|
| `cp a b` | Copia archivo | Versionado |
| `cp a dir/` | Copia a carpeta | Staging |
| `cp -r dir1 dir2` | Copia carpeta | Data lake |
| `cp -v` | Muestra progreso | Debug |
| `cp -p` | Preserva metadata | Auditoría |
| `cp -u` | Copia incremental | ETL |
| `cp *.csv dir/` | Copia por patrón | Batch |

In [ ]:
# Versionar un dataset antes de limpiarlo
!cp raw/ventas_2024.csv raw/ventas_2024_backup.csv
!ls raw

In [ ]:
# Copiar a una carpeta mostrando lo que hace (-v)
!cp -v raw/ventas_2025.csv processed/

In [ ]:
# Copiar por patrón y preservando metadata (fecha original)
!mkdir -p backup
!cp -pv raw/*.csv backup/
!ls -l backup

In [ ]:
# Copia incremental: solo copia si el origen es más nuevo que el destino
!cp -uv raw/*.csv backup/     # no debería copiar nada (ya están al día)

In [ ]:
# Copiar una carpeta completa (clonar un data lake local)
!cp -r logs logs_copia
!ls -R logs_copia

### ✏️ Ejercicio 4.1
1. Crea la carpeta `sandbox` y copia dentro **toda** la carpeta `raw` (entorno de pruebas).
2. Crea una versión `raw/ventas_2025_v1.csv` a partir de `raw/ventas_2025.csv` **preservando** su metadata.
3. Agrega una línea nueva a `raw/ventas_2025.csv` con `echo "7,2025-05-01,ana@mail.com,Mouse,48,OK" >> raw/ventas_2025.csv` y luego ejecuta una **copia incremental verbosa** de `raw/*.csv` hacia `backup/`. ¿Qué archivo(s) se copiaron y por qué?

In [ ]:
# Tu código aquí

## 5. `mv` — Mover y renombrar
Versionar datasets y marcar cambios en data limpia.

| Comando | ¿Qué hace? | Uso DS/DE |
|---|---|---|
| `mv a b` | Renombra | Versionado |
| `mv a dir/` | Mueve archivo | Organización |
| `mv *.csv dir/` | Mueve por patrón | Batch |
| `mv -i` | Confirmación | Seguridad |
| `mv -f` | Forzado | Automatización |
| `mv -v` | Verbose | Logging |

> 💡 En Colab, `-i` (interactivo) no puede esperar tu respuesta; en scripts automáticos se usa `-f` o `-n` (no sobrescribir).

In [ ]:
# Renombrar: marcar el dataset como limpio
!mv -v processed/ventas_2025.csv processed/ventas_2025_clean.csv
!ls processed

In [ ]:
# Mover por patrón a una carpeta de staging
!mkdir -p staging
!mv -v tmp/*.csv staging/
!ls staging tmp

### ✏️ Ejercicio 5.1
1. Renombra `raw/Clientes.CSV` a `raw/clientes.csv` (nombres en minúscula = buena práctica).
2. Crea la carpeta `logs/2025` y **mueve** allí todos los `.log` de `logs/` (no los de `archivo_2024`) mostrando lo que se movió.
3. Renombra `raw/ventas_2024_backup.csv` a `backup/ventas_2024_$(date +%Y%m%d).csv` (versionado con fecha).

In [ ]:
# Tu código aquí

## 6. `find` — Localizar archivos por nombre, tipo, tamaño, fecha…

| Comando | ¿Qué hace? | Uso DS/DE |
|---|---|---|
| `find . -name` | Buscar por nombre | Data discovery |
| `find . -iname` | Ignora mayúsculas | Consistencia |
| `find . -type f/d` | Archivos / carpetas | Auditoría |
| `find . -size` | Tamaño | Costos |
| `find . -mtime` | Fecha de modificación | Jobs |
| `find ... -exec` | Ejecutar comandos | ETL |
| `find ... -delete` | Borrar | Limpieza |

In [ ]:
# Todos los CSV del laboratorio
!find . -name "*.csv"

In [ ]:
# -iname: encuentra .csv y .CSV
!find . -iname "*.csv"

In [ ]:
# Solo carpetas
!find . -type d

In [ ]:
# Archivos de más de 500 KB (controlar costos de almacenamiento)
!find . -type f -size +500k -exec ls -lh {} \;

In [ ]:
# Archivos modificados hace más de 30 días
!find . -type f -mtime +30

In [ ]:
# -exec: contar líneas de cada CSV encontrado
!find raw -name "*.csv" -exec wc -l {} \;

In [ ]:
# -delete: limpiar temporales (primero SIEMPRE listar, luego borrar)
!find . -name "*.tmp"
!find . -name "*.tmp" -delete
!find . -name "*.tmp" | wc -l

### ✏️ Ejercicio 6.1
1. Encuentra todos los archivos `.log` del laboratorio (en cualquier subcarpeta).
2. Encuentra los archivos **menores a 1 KB** dentro de `raw`.
3. Lista los logs con **más de 30 días** de antigüedad y **cópialos** a una carpeta `historico/` usando `-exec`.
4. Usa `find` + `-exec grep` para mostrar qué archivos `.csv` contienen la palabra `PENDIENTE` (pista: `grep -l`).

In [ ]:
# Tu código aquí

## 7. `rm` — Eliminar archivos y directorios
Borrar outputs temporales, limpiar intermedios de ETL, eliminar duplicados.

| Comando | ¿Qué hace? | Uso DS/DE |
|---|---|---|
| `rm file.txt` | Borra archivo | Limpieza manual |
| `rm *.csv` | Borra por patrón | Outputs antiguos |
| `rm -r dir/` | Borra carpeta | Staging |
| `rm -f file.txt` | Fuerza (sin error si no existe) | Scripts |
| `rm -rf dir/` | Todo | Reset de entorno |
| `rm -i file.txt` | Confirma | Aprendizaje |

> ⚠️ En Linux **no hay papelera**: lo que borras con `rm` no se recupera. Antes de un `rm` con patrón, ejecuta primero `ls` con el mismo patrón.

In [ ]:
# Borrar un archivo
!rm staging/staging_parcial.csv
!ls staging

In [ ]:
# rm sin -f falla si el archivo no existe; con -f no (útil en scripts)
!rm no_existe.txt
!rm -f no_existe.txt && echo "Sin error con -f"

In [ ]:
# Borrar una carpeta completa
!rm -r logs_copia
!ls

### ✏️ Ejercicio 7.1
1. Lista y luego borra el archivo `raw/big_dataset.txt`.
2. Borra la carpeta `staging` (que ahora está vacía) con su contenido.
3. Elimina la carpeta `sandbox` completa **sin que falle aunque no exista**.
4. Explica con un comentario: ¿por qué es peligroso ejecutar `rm -rf $CARPETA/` si la variable `CARPETA` está vacía?

In [ ]:
# Tu código aquí

## 8. Sesión 1.2 — Scripts de Shell y Automatización
Un **script** es un archivo de texto con comandos que la shell ejecuta en orden. En vez de escribir 10 comandos, los guardas en un `.sh` y los ejecutas de un solo golpe.

**Anatomía de un script profesional:**
1. **Shebang** `#!/bin/bash` → primera línea: indica qué intérprete usar.
2. **Comentarios** `#` → explican qué limpieza haces al dataset.
3. **Comandos de shell** → `cd`, `ls`, `mkdir`, `grep`, `awk`, …

In [ ]:
%%writefile /content/lab_shell/reporte_logs.sh
#!/bin/bash
# =========================================================
# Script: reporte_logs.sh
# Objetivo: generar un reporte de calidad de los logs ETL
# =========================================================

# Directorio base del laboratorio
BASE=/content/lab_shell
cd "$BASE"

# Crear carpeta de reportes si no existe
mkdir -p reportes

echo "===== REPORTE DE LOGS ($(date '+%Y-%m-%d %H:%M')) ====="

# Contar errores (sin importar mayúsculas) en todos los logs
echo "Total de errores: $(grep -ri error logs/ | wc -l)"

# Contar advertencias
echo "Total de warnings: $(grep -r WARN logs/ | wc -l)"

# Guardar el detalle de errores en un archivo
grep -rin error logs/ > reportes/detalle_errores.txt
echo "Detalle guardado en reportes/detalle_errores.txt"

In [ ]:
# Dar permisos de ejecución y ejecutar
!chmod +x /content/lab_shell/reporte_logs.sh
!/content/lab_shell/reporte_logs.sh

In [ ]:
!cat /content/lab_shell/reportes/detalle_errores.txt

### ✏️ Ejercicio 8.1 — Tu primer pipeline en bash
Crea un script `/content/lab_shell/limpieza.sh` que:
1. Tenga **shebang** y **comentarios** explicando cada paso.
2. Cree la carpeta `processed/` si no existe.
3. Genere `processed/ventas_ok.csv` con el **encabezado** + solo las ventas con estado `OK` de **ambos** años.
4. Genere `processed/ventas_con_problemas.csv` con las filas cuyo estado sea `ERROR`/`error` o `PENDIENTE`.
5. Imprima cuántas filas tiene cada archivo generado.
6. Borre la carpeta `tmp/` al final (limpieza de intermedios).

Luego dale permisos con `chmod +x` y ejecútalo.

In [ ]:
%%writefile /content/lab_shell/limpieza.sh
#!/bin/bash
# Tu script aquí

In [ ]:
# Ejecuta tu script aquí

## 9. 🏆 Reto integrador
Escribe un script `organizar_datalake.sh` que reciba **como argumento** el año a procesar (ej. `./organizar_datalake.sh 2025`) y:
1. Cree la estructura `datalake/<año>/{bronze,silver,gold}`.
2. Copie (preservando metadata) el archivo `raw/ventas_<año>.csv` a `bronze/`.
3. En `silver/` deje una versión **sin encabezado** y **sin filas con error** (ignorando mayúsculas).
4. En `gold/resumen.txt` escriba: total de filas válidas y el número de ventas por producto. *(Pista: `cut -d, -f4 | sort | uniq -c`)*
5. Use `find` para listar al final todo lo generado en `datalake/<año>`.

> Pista: dentro de un script, el primer argumento se lee con `$1`.

In [ ]:
%%writefile /content/lab_shell/organizar_datalake.sh
#!/bin/bash
# Tu script aquí

In [ ]:
# Ejecuta tu reto aquí

---
## 🎯 Fin de los ejercicios
Las soluciones están en el notebook **`Capitulo1_Soluciones.ipynb`**. Intenta resolver todo antes de revisarlas.